In [2]:
import re
from pydriller import Repository

# 1. Setup Inputs
issue_ids = ["CAMEL-180", "CAMEL-321", "CAMEL-1818", "CAMEL-3214", "CAMEL-18065"]
# Adjust this path if your script is in a different folder than the repo
local_repo_path = "camel" 

# Create a regex pattern to match exact issue IDs (prevents CAMEL-180 matching CAMEL-18065)
# \b creates a "word boundary" so it only matches the exact ID
# We added re.IGNORECASE to catch camel-180, Camel-321, etc.
pattern = re.compile(r'\b(?:' + '|'.join(issue_ids) + r')\b', re.IGNORECASE)

# 2. Tracking Variables
total_commits = 0
unique_files = set()
total_dmm_score = 0.0

print("Analyzing local repository... (This may take a moment)")

# 3. Traverse the Local Repository
for commit in Repository(local_repo_path).traverse_commits():
    
    # Check if the commit message contains any of the exact issue IDs
    if pattern.search(commit.msg):
        total_commits += 1
        
        # --- Task 1: Track Unique Files Modified ---
        for modified_file in commit.modified_files:
            # Files can be added, modified, or deleted. We grab the path.
            # new_path exists for added/modified, old_path exists for deleted
            file_path = modified_file.new_path or modified_file.old_path
            if file_path:
                unique_files.add(file_path)
        
        # --- Task 2: Calculate DMM Metrics ---
        # PyDriller automatically calculates these, but may return None if 
        # the commit doesn't contain supported code (e.g., only text files)
        size = commit.dmm_unit_size
        complexity = commit.dmm_unit_complexity
        interfacing = commit.dmm_unit_interfacing
        
        if size is not None and complexity is not None and interfacing is not None:
            # Average of the three DMM properties for this specific commit
            commit_dmm = (size + complexity + interfacing) / 3.0
            total_dmm_score += commit_dmm

# 4. Calculate Final Global Averages
if total_commits > 0:
    # Total unique files across all found commits / total commits
    avg_unique_files = len(unique_files) / total_commits
    
    # Sum of DMM scores / total commits
    avg_dmm_metrics = total_dmm_score / total_commits
    
    # Print exactly as requested
    print("\n--- Results ---")
    print(f"Total commits analyzed: {total_commits}")
    print(f"Average number of files changed: {avg_unique_files:.2f}")
    print(f"Average DMM metrics: {avg_dmm_metrics:.2f}")
else:
    print("\nNo commits found matching those Issue IDs. Please check your repo path.")

Analyzing local repository... (This may take a moment)

--- Results ---
Total commits analyzed: 2
Average number of files changed: 14.50
Average DMM metrics: 0.70
